# Workshop: real-gold 4D-STEM — browse, BF, DF, probe, DPC (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/bobleesj/a05a90185c6cddbb331342cae6d7e9c1/berk_workshop_v1.ipynb)

Load a real 4D-STEM gold dataset from Hugging Face, browse it interactively,
compute bright field / dark field, find the probe center, and visualize DPC —
all in your browser, on Colab's free T4 GPU. No local install, no `quantem.live`.

Two installs only: `quantem.widget` (TestPyPI prerelease) and `quantem`
(from the `berk-workshop` branch on `bobleesj/quantem`). Everything runs on
torch on the GPU.

**Total runtime: 2–3 minutes** (install dominates).

In [1]:
!pip install -q --pre --extra-index-url https://test.pypi.org/simple/ quantem.widget huggingface_hub
!pip install -q git+https://github.com/bobleesj/quantem.git@berk-workshop

In [2]:
import quantem as em
import quantem.widget
import torch

print("quantem        ", em.__version__)
print("quantem.widget ", quantem.widget.__version__)
print("torch          ", torch.__version__)
print("cuda available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU runtime)")

quantem         0.1.8
quantem.widget  0.0.1
torch           2.10.0+cu130
cuda available: True NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [3]:
# Download a pre-binned NumPy bundle from the public Hugging Face dataset and
# wrap it as a torch tensor on the GPU. The 4D shape is (scan_row, scan_col,
# k_row, k_col) = (512, 512, 24, 24); detector binned 8x from the original 192x192
# to keep the Colab download small (~300 MB).
import os, json
import numpy as np
from huggingface_hub import snapshot_download

folder = snapshot_download("bobleesj/quantem-data", repo_type="dataset",
                           allow_patterns=["4dstem/gold_512_npy_bin8/*"])
asset = os.path.join(folder, "4dstem", "gold_512_npy_bin8")
data = np.load(os.path.join(asset, "data.npy"))           # (512, 512, 24, 24) uint16
meta = json.load(open(os.path.join(asset, "meta.json")))

device = "cuda" if torch.cuda.is_available() else "cpu"
t = torch.from_numpy(data).to(device)
dset = em.core.datastructures.Dataset4dstem.from_tensor(
    t, sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
print(f"dataset on {dset.tensor.device}: shape {dset.shape}, dtype {dset.tensor.dtype}")
print(f"sampling {meta['sampling']} {meta['units']}")
print(f"optics: {meta['voltage_kV']} kV, probe {meta['probe_semiangle_mrad']} mrad, CL {meta['camera_length_mm']} mm")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

dataset on cuda:0: shape (512, 512, 24, 24), dtype torch.uint16
sampling [0.5, 0.5, 3.68, 3.68] ['A', 'A', 'mrad', 'mrad']
optics: 300 kV, probe 30 mrad, CL 91 mm


## Step 1 — Browse the 4D-STEM dataset interactively

Drag the scan cursor in the left panel. The CBED on the right updates live.
This is your real-time bright-field / dark-field — pixels INSIDE the central
disk are BF, OUTSIDE are DF. The widget renders in your browser via WebGPU
(works in Chrome on free Colab).

In [4]:
quantem.widget.Show4DSTEM(dset)

  to cuda:0: 0.00s (0.3 GB)
  auto_detect_center: 0.08s
  virtual image + frame: 0.00s
Show4DSTEM: 512x512x24x24 cuda:0, 0.24s total


Show4DSTEM(shape=(512, 512, 24, 24), sampling=(0.5 A, 3.68 mrad), pos=(256, 256), title='gold_512_npy_bin8')

## Step 2 — Detector geometry + aperture center

Detector size comes from the dataset; aperture center is hardcoded to the
geometric center of the detector (for the binned data here it lands close to
the real beam center).

In [6]:
H, W = dset.tensor.shape[-2:]          # detector dims
cy, cx = H / 2, W / 2                  # hardcoded — geometric center
device = dset.tensor.device

row = torch.arange(H, device=device, dtype=torch.float32)[:, None]
col = torch.arange(W, device=device, dtype=torch.float32)[None, :]

print(f"detector {H}x{W}, aperture center hardcoded to (row={cy}, col={cx})")

probe center: row=11.289, col=11.503   (detector 24x24, geometric center ~12.0, ~12.0)


## Step 3 — Bright field

Aperture mask at the probe center → for every scan position, sum the detector
pixels INSIDE the BF disk. One torch reduction on the GPU; renders as an
interactive `Show2D` widget.

In [ ]:
BF_RADIUS_PX = 6.0   # ~30 mrad on the bin8 detector

rr, cc = torch.meshgrid(row.squeeze(), col.squeeze(), indexing="ij")
r_from_center = ((rr - cy) ** 2 + (cc - cx) ** 2).sqrt()
bf_mask = (r_from_center <= BF_RADIUS_PX).float()
df_mask = 1.0 - bf_mask                     # for the next step

data_f = dset.tensor.to(torch.float32)
bf = (data_f * bf_mask).sum(dim=(-2, -1)).cpu().numpy()

quantem.widget.Show2D(
    bf,
    title="Bright field",
    sampling=meta["sampling"][:2],
    units=meta["units"][:2],
    cmap="gray",
)

## Step 4 — Dark field

Same data, opposite mask — sum the detector pixels OUTSIDE the BF disk.

In [ ]:
df = (data_f * df_mask).sum(dim=(-2, -1)).cpu().numpy()

quantem.widget.Show2D(
    df,
    title="Dark field",
    sampling=meta["sampling"][:2],
    units=meta["units"][:2],
    cmap="gray",
)

## Step 5 — Center of mass (DPC) per scan position

For every scan position, the centroid of its diffraction pattern shifts when a
local electric field deflects the probe. Compute that centroid in torch on the
GPU. Three interactive `Show2D` widgets follow:

- **CoM row (qx)** — signed deflection along the rows of the detector.
- **CoM col (qy)** — signed deflection along the columns.
- **|CoM|** — total deflection magnitude (rotation-invariant).

In [ ]:
H, W = data_f.shape[-2:]
qx = torch.arange(H, device=data_f.device, dtype=torch.float32)[:, None]
qy = torch.arange(W, device=data_f.device, dtype=torch.float32)[None, :]

total_per_dp = data_f.sum(dim=(-2, -1))                  # (scan_r, scan_c)
com_row = (data_f * qx).sum(dim=(-2, -1)) / total_per_dp
com_col = (data_f * qy).sum(dim=(-2, -1)) / total_per_dp

# Detrend so the colormap is zero-centered on the signed deflection.
com_row -= com_row.mean()
com_col -= com_col.mean()

com_mag = (com_row ** 2 + com_col ** 2).sqrt()           # |CoM|, always >= 0

print(f"CoM row range [{com_row.min().item():.4f}, {com_row.max().item():.4f}] px")
print(f"CoM col range [{com_col.min().item():.4f}, {com_col.max().item():.4f}] px")
print(f"|CoM| max     {com_mag.max().item():.4f} px")

quantem.widget.Show2D(
    com_row.cpu().numpy(),
    title="CoM row (qx) — DPC",
    sampling=meta["sampling"][:2],
    units=meta["units"][:2],
    cmap="RdBu_r",
)

In [ ]:
quantem.widget.Show2D(
    com_col.cpu().numpy(),
    title="CoM col (qy) — DPC",
    sampling=meta["sampling"][:2],
    units=meta["units"][:2],
    cmap="RdBu_r",
)

In [ ]:
quantem.widget.Show2D(
    com_mag.cpu().numpy(),
    title="|CoM| (total deflection)",
    sampling=meta["sampling"][:2],
    units=meta["units"][:2],
    cmap="magma",
)

## What you just did

1. Loaded real 4D-STEM gold from Hugging Face → torch tensor on the Colab T4.
2. Browsed it with `Show4DSTEM` and found the probe center via CoM of the mean DP.
3. Bright field via aperture mask × per-scan sum, rendered with `Show2D`.
4. Dark field via the complementary mask.
5. DPC / CoM maps: row, column, and total magnitude — three interactive `Show2D` widgets.

Every reduction ran on the GPU in milliseconds; every visualization is an
interactive WebGPU widget (zoom, contrast, scale bar).

## Try next

- Swap `gold_512_npy_bin8` → `gold_512_npy_bin4` for a finer (4× larger) detector.
- Change `BF_RADIUS_PX` to grow/shrink the BF disk.
- Detrend DPC differently (subtract a plane fit) to surface smaller features.

v2 will add iterative ptychography (`PtychoLite`) on the same data.